# 🧪 Sesión 34 · Integración Frontend: Fetch API y JavaScript Asíncrono

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Cómo pide datos tu página sin recargarse?

---

## 🖥️ Así se verá el resultado final

```javascript
// consola del navegador
const respuesta = await fetch("/api/libros");
if (!respuesta.ok) throw new Error(`HTTP ${respuesta.status}`);
const datos = await respuesta.json();
console.log(datos);
// { libros: [ { id: 1, titulo: "Rayuela" } ] }

<!-- HTML renderizado -->
<ul id="lista">
  <li>Rayuela — 1963</li>
</ul>

<!-- petición autenticada -->
fetch("/api/perfil", {
  headers: { Authorization: `Bearer ${token}` }
});
```

_fetch no falla en un 404: la promesa se resuelve. Debes revisar response.ok._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Consumir una API con fetch y async/await.
- Validar response.ok antes de leer el JSON.
- Enviar el token en el encabezado Authorization.

## ✅ Criterios de aceptación

El verificador comprobará estos criterios sobre los archivos que produzcas:

- [ ] El cliente usa fetch, valida la respuesta y autentica
- [ ] La página carga el cliente

## 🧩 El reto

Escribe el cliente JavaScript que consume la API de libros, maneja errores y envía el token cuando corresponde.

**Requisitos del contrato**

- La función de carga usa async/await y fetch.
- Se valida response.ok antes de parsear el JSON.
- Se usa JSON.stringify al enviar un cuerpo.
- El token se envía como Authorization: Bearer.
- El resultado se muestra en el elemento #lista.

> **Pista conceptual:** Separar la petición de la renderización te permite probar cada parte. `fetch` devuelve una respuesta; `response.json()` devuelve otra promesa.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion34.md` con una captura de la consola y la lista renderizada.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `index.html`, `app.js`


### 📄 `index.html`

Página que contiene la lista.

In [2]:
%%writefile index.html
<!DOCTYPE html>
<html lang="es">
<head><meta charset="UTF-8"><title>Biblioteca</title></head>
<body>
  <h1>Biblioteca</h1>
  <ul id="lista"></ul>
  <script src="app.js"></script>
</body>
</html>


Writing index.html


### 📄 `app.js`

Cliente Fetch.

In [4]:
%%writefile app.js
const API = "/api/libros";
const lista = document.getElementById("lista");

// El token se obtiene al hacer login (POST /api/token) y se guarda en memoria.
let token = null;

function cabeceras(conJson = false) {
  const h = {};
  if (conJson) h["Content-Type"] = "application/json";
  if (token) h["Authorization"] = "Bearer " + token;
  return h;
}

async function cargarLibros() {
  try {
    const respuesta = await fetch(API, { headers: cabeceras() });
    if (!respuesta.ok) {
      throw new Error("Error " + respuesta.status + " al cargar los libros");
    }
    const datos = await respuesta.json();
    mostrarLibros(datos.libros);
  } catch (error) {
    mostrarError(error.message);
  }
}

async function crearLibro(titulo, anio) {
  const respuesta = await fetch(API, {
    method: "POST",
    headers: cabeceras(true),
    body: JSON.stringify({ titulo: titulo, anio: Number(anio) }),
  });
  if (!respuesta.ok) {
    throw new Error("Error " + respuesta.status + " al crear el libro");
  }
  return await respuesta.json();
}

function mostrarLibros(libros) {
  lista.replaceChildren();
  if (libros.length === 0) {
    const vacio = document.createElement("li");
    vacio.textContent = "No hay libros todavía.";
    lista.appendChild(vacio);
    return;
  }
  for (const libro of libros) {
    const item = document.createElement("li");
    // textContent evita inyectar HTML si un título trae etiquetas.
    item.textContent = libro.titulo + " (" + libro.anio + ")";
    lista.appendChild(item);
  }
}

function mostrarError(mensaje) {
  lista.replaceChildren();
  const item = document.createElement("li");
  item.textContent = mensaje;
  lista.appendChild(item);
}

function construirFormulario() {
  const contenedor = document.createElement("div");
  const titulo = document.createElement("input");
  titulo.placeholder = "Título";
  const anio = document.createElement("input");
  anio.type = "number";
  anio.placeholder = "Año";
  const boton = document.createElement("button");
  boton.textContent = "Agregar";

  boton.addEventListener("click", async () => {
    try {
      await crearLibro(titulo.value, anio.value);
      titulo.value = "";
      anio.value = "";
      await cargarLibros();
    } catch (error) {
      mostrarError(error.message);
    }
  });

  contenedor.append(titulo, anio, boton);
  lista.before(contenedor);
}

construirFormulario();
cargarLibros();


Overwriting app.js


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [5]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.js", regex=["async\\s+function|async\\s*\\(", "fetch\\(", "await\\s+.*\\.json\\(\\)", "\\.ok", "Authorization", "Bearer", "JSON\\.stringify", "lista"], descripcion="El cliente usa fetch, valida la respuesta y autentica"))
resultados.append(revisar_archivo("index.html", exigir=["id=\"lista\"", "app.js"], descripcion="La página carga el cliente"))
resumen(resultados)


✅ El cliente usa fetch, valida la respuesta y autentica
✅ La página carga el cliente
------------------------------------------------------------------------------
RESULTADO: 2/2 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion34.md` a `auditorias/auditoria_sesion34.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
